# Example: Alternative initial plasma flux guesses

---

In both **static forward** and **static inverse** Grad-Shafranov (GS) solves, FreeGSNKE's Newton-Krylov solver starts from an initial guess for the plasma poloidal flux $\psi_{p}$. Recall tht total poloidal flux is $\psi = \psi_p + \psi_c$ where $\psi_c$ is the flux from the conducting structures. 

While the default Gaussian guess is sufficient for many equilibria, a physically motivated initial guess that better approximates the expected plasma location, elongation, and plasma current can reduce the number of Newton-Krylov iterations and improve convergence for some configurations.

This notebook demonstrates how to configure and compare FreeGSNKE's different initial flux guess constructors on a **MAST-U-like tokamak**.

**Note:** It is recommended to review `example01a` (static inverse) and `example02a` (static forward) before running this notebook, as shared setup details are kept brief here.

### Import packages

In [ ]:
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np

from freegsnke import GSstaticsolver, build_machine, equilibrium_update
from freegsnke.equilibrium import PsiGuessGaussian, PsiGuessParabolicJtor
from freegsnke.jtor_update import ConstrainPaxisIp

### Create the machine object

Build the MAST-U-like tokamak from the packaged machine configuration pickle files. 

In [ ]:
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

### Instantiate an equilibrium and profile object

In [ ]:
# initialise equilibrium and profiles
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,   # must be of the form (2**n + 1)
    ny=129,  # must be of the form (2**n + 1)
)

profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8e3,    # profile object
    Ip=6e5,       # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

### 1. Gaussian-shape plasma flux (`PsiGuessGaussian`)

The default initial guess in the `eq` object uses a Gaussian-shaped plasma flux distribution defined in **normalised grid coordinates** $(x, y) \in [0, 1]^2$:

$$
\psi_p(x, y) = \exp\!\left(C - \left(|x - x_c|^p + |y - y_c|^p\right)\right),
$$

with $\psi_p = 0$ enforced on the computational domain boundary. 

The parameters for this guess are supplied via `gpars=(xc, yc, C, p)` (defaulting to `(0.5, 0.5, 0, 2)`).

In [ ]:
# initialise the class 
plasma_psi_gaussian = PsiGuessGaussian(gpars=(0.5, 0.5, 0.0, 2.0))

### 2. Parabolic-shaped current density guess (`PsiGuessParabolicJtor`)

This alternative guess assumes the core plasma boundary can be approximated by an ellipse with centre $(h, k)$ and semi-axes $(a, b)$, where the elliptic coordinate $E(R, Z)$ satisfies $E(h, k) = 0$ and $E(R, Z) = 1$ on the boundary. 

An approxaimte parabolic toroidal current density profile

$$
J_{\text{tor}}(R, Z) = \hat{J}\left(1 - E(R, Z)^i\right)^j
$$

is scaled via $\hat{J}$ to match the prescribed plasma current $I_p$, and a single linear Grad-Shafranov solve yields the initial $\psi_p$ guess (see [Wai et al., (2026)](https://iopscience.iop.org/article/10.1088/1741-4326/ae203c/meta)).

The parameters for this type of gues can be constructed in three ways:
1. **Direct ellipse parameters**: `PsiGuessParabolicJtor(h, k, a, b, Ip, psi_scale=1.0)`.
2. **X-point and midplane coordinates**: `PsiGuessParabolicJtor.from_xpoint_and_midplane(Rx, Zx, Ri, Ro, Zm, Ip)`, which sets $h = (R_x + R_i + R_o)/3$.
3. **LCFS points**: `PsiGuessParabolicJtor.from_separatrix_points(Ip, *points)`, which fits $(h, k, a, b)$ to four or more points on a user-provided plasma boundary using ordinary least squares (OLS).

Depending on which information you have available prior to a solve, you can choose either of the three above. 

In [ ]:
# let's construct all three types below

# type 1 (direct ellipse parameterisation)
plasma_psi_parabolic_standard = PsiGuessParabolicJtor(
    h = (min(tokamak.limiter.R)+max(tokamak.limiter.R))/2,  # centre of ellipse (R)
    k = 0.0,          # centre of ellipse (Z)
    a = 0.5,          # roughly the minor radius of the plasma
    b = 0.5 * 2,      # roughly the minor radius times the elongation of expected plasma
    Ip = profiles.Ip, # plasma current (to scale the guess)
    psi_scale = 1.0   # might need to vary this default
    )

# type 2 (use approximate up/down symmetric X-point locations and midplane)
Rx, Zx = 0.6, 1.1
Ri, Ro, Zm = 0.34, 1.4, 0.0

plasma_psi_parabolic_xpoints = PsiGuessParabolicJtor.from_xpoint_and_midplane(
        Rx=Rx, Zx=Zx, Ri=Ri, Ro=Ro, Zm=Zm, Ip=profiles.Ip,
)

# type 3 (use approximate LCFS points)
plasma_psi_parabolic_lcfs = PsiGuessParabolicJtor.from_separatrix_points(
        profiles.Ip,
        (Rx, Zx), (Rx, -Zx), (Ri, Zm), (Ro, Zm),
)

Let us visualise four of these initial $\psi_p$ guesses side by side inside the MAST-U machine.

In [ ]:
initial_cases = [
    ("Default Gaussian", plasma_psi_gaussian),
    ("Parabolic (direct ellipse)", plasma_psi_parabolic_standard),
    ("Parabolic (X-pt & midplane)", plasma_psi_parabolic_xpoints),
    ("Parabolic (LCFS)", plasma_psi_parabolic_lcfs),
]

# plot
fig, axes = plt.subplots(1, 4, figsize=(14, 6), dpi=80, sharex=True, sharey=True)
for ax, (title, case) in zip(axes, initial_cases):
    ax.grid(True, alpha=0.5)
    ax.set_aspect("equal")
    tokamak.plot(axis=ax, show=False)
    ax.plot(tokamak.wall.R, tokamak.wall.Z, "k", linewidth=1.2)
    case.plasma_psi = case(eq=eq)
    levels = np.linspace(np.min(case.plasma_psi), np.max(case.plasma_psi), 25)
    ax.contour(eq.R, eq.Z, case.plasma_psi, levels=levels)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0.1, 2.15)
    ax.set_ylim(-2.25, 2.25)

plt.tight_layout()
plt.show()

If we now initialise the coil currents inside the `eq` object, we can construct the coil flux and add it to the plasma flux guesses above. This will show us what the total flux passes to the static solver looks like during the first Newton-Krylov iteration. 

In [ ]:
# set coil currents
with open("data/simple_diverted_currents_PaxisIp.pk", "rb") as f:
    diverted_currents = pickle.load(f)

for key, val in diverted_currents.items():
    eq.tokamak.set_coil_current(coil_label=key, current_value=val)
    
# flux from the coils
psi_coils = eq.tokamak.calcPsiFromGreens(pgreen=eq._pgreen)

In [ ]:
# plot just flux from coils
fig1, ax1 = plt.subplots(1, 1, figsize=(3, 6), dpi=80)
ax1.grid(True, which='both')
ax1.plot(tokamak.wall.R, tokamak.wall.Z, 'k')
eq.tokamak.plot(axis=ax1, show=False)
ax1.contour(eq.R, eq.Z, psi_coils, levels=25)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

Notice how the total flux maps shown below are starting to look like Grad-Shafranov equilibria! The solver can then work with these guesses to converge to tolerance (as we'll see next).

In [ ]:
# now plot flux from coils with the different plasma flux guesses
fig, axes = plt.subplots(1, 4, figsize=(14, 6), dpi=80, sharex=True, sharey=True)
for ax, (title, case) in zip(axes, initial_cases):
    ax.grid(True, alpha=0.5)
    ax.set_aspect("equal")
    tokamak.plot(axis=ax, show=False)
    ax.plot(tokamak.wall.R, tokamak.wall.Z, "k", linewidth=1.2)
    total_flux = case.plasma_psi + psi_coils
    levels = np.linspace(np.min(total_flux), np.max(total_flux), 25)
    ax.contour(eq.R, eq.Z, total_flux, levels=levels)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0.1, 2.15)
    ax.set_ylim(-2.25, 2.25)

plt.tight_layout()
plt.show()

---
### Static forward solve comparison

To quantify the effect of the initial plasma flux guess on solver convergence, we solve the same diverted forward equilibrium from each initial guess. 

We can store the GS residual at each iteration for comparison.

In [ ]:
# initialise solver object
solver = GSstaticsolver.NKGSsolver(eq, gs_operator_order=4)

In [ ]:
# run each of the solves
results = {}
for label, case in initial_cases:

    # initial guess
    eq.plasma_psi = case.plasma_psi
    
    # timer
    t0 = time.perf_counter()
    
    # solve
    solver.solve(
        eq=eq,
        profiles=profiles,
        target_relative_tolerance=1e-9,
        verbose=False,
    )
    elapsed = time.perf_counter() - t0

    # store results
    results[label] = {
        "psi_final": eq.plasma_psi.copy(),
        "residual": np.asarray(solver.norm_rel_change),
        "initial_res": solver.initial_rel_residual,
        "iters": len(solver.norm_rel_change) - 1,
        "time_taken": elapsed,
    }
    print(
        f"{label:28s} | iters: {results[label]['iters']:2d} "
        f"| init residual: {results[label]['initial_res']:.3e} "
        f"| final residual: {results[label]['residual'][-1]:.3e} "
        f"| time: {elapsed:.3f} s"
    )

Now let's plot the history of the residual from each solve to see the iterations taken. 

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(4.5, 4), dpi=90)

for label, res in results.items():
    ax1.semilogy(
        np.arange(len(res["residual"])),
        res["residual"],
        marker="o",
        markersize=4,
        label=label,
    )
ax1.axhline(1e-9, color="k", linestyle="--", linewidth=1.0, label="Convergence tolerance")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Relative GS residual")
ax1.grid(True, which="both", alpha=0.4)
ax1.legend(fontsize=8)

plt.tight_layout()
plt.show()

While the default Gaussian guess is acceptable for most use cases, convergence can be **accelerated** using the parabolic Jtor guess if certain features of the plasma core shape are known before simulation!